```{=latex}
\usepackage{hyperref}
\usepackage{graphicx}
\usepackage{listings}
\usepackage{textcomp}
\usepackage{fancyvrb}

\newcommand{\passthrough}[1]{\lstset{mathescape=false}#1\lstset{mathescape=false}}
\newcommand{\tightlist}{}
```

```{=latex}
\title{Back Off and Give Up}
\author{Moshe Zadka -- https://cobordism.com}
\date{}

\begin{document}
\begin{titlepage}
\maketitle
\end{titlepage}

\frame{\titlepage}
```

```{=latex}
\begin{frame}
\frametitle{Acknowledgement of Country}

Hayward (in San Francisco Bay Area)

Ancestral homeland of the Ohlone people

\end{frame}
```

I live in Hayward,
in the San Francisco Bay Area.
I wish to acknowledge it as the
ancestral homeland
of the
Ohlone people.

```{=latex}
\begin{frame}
\frametitle{The Secret to Resilient Systems}

\pause

Quit

\end{frame}
```

What if I told you that the secret to building resilient systems is learning when to quit? This might sound counterintuitive - we're taught that persistence is a virtue. But in distributed systems, sometimes giving up is exactly the right strategy. Sometimes, trying harder makes things worse.

```{=latex}
\begin{frame}
\frametitle{A Tale of Two Worlds: Premise}

Recipe recommendation site with ML backend

\end{frame}
```

```{=latex}
\begin{frame}
\frametitle{A Tale of Two Worlds: World 1}

Max backoff = 1 second

\end{frame}
```

```{=latex}
\begin{frame}
\frametitle{A Tale of Two Worlds: World 2}

Max backoff = 10 minutes

\end{frame}
```

Let me tell you a story about a recipe recommendation site with an ML backend. During a model rollout, nodes started becoming flaky. In one world - let's call it World 1 - the max backoff was set to 1 second. The front-ends hammered the ML nodes relentlessly, causing a 3-hour complete outage that made the news. But in World 2, where the max backoff was 10 minutes, the system recovered gracefully with just minor degradation that nobody even noticed. One configuration parameter. Two completely different outcomes.

```{=latex}
\begin{frame}[fragile]
\frametitle{Naive Retries}

The problem

\end{frame}
```

Part 1: The Problem with Naive Retries

```{=latex}
\begin{frame}[fragile]
\frametitle{Naive Retry: The Classic Mistake}

\begin{verbatim}
import time
import requests

def get_data(url):
    while True:
        try:
            response = requests.get(url, timeout=5)
            return response.json()
        except requests.RequestException:
            time.sleep(1)
\end{verbatim}

\end{frame}
```

Here's code we've all written at some point. Simple retry loop - if it fails, wait a second and try again. Forever. Infinite retries, fixed delay, no backoff, no jitter, and no maximum retry limit. When the service comes back, every client hits it at the same time.

```{=latex}
\begin{frame}
\frametitle{Thundering Herd}

\pause

Recover

\pause

Crash

\end{frame}
```

This is the thundering herd problem. Service goes down at time zero. A thousand clients start retrying every second. Service recovers after 60 seconds. What happens? BOOM - a thousand simultaneous connections hit it all at once. The service that just recovered immediately crashes again. This is exactly what happened in our recipe site story.

```{=latex}
\begin{frame}
\frametitle{Retry Storms}

Frontend $\rightarrow$ API Gateway $\rightarrow$ Payment Service

\vspace{1em}

Each layer: 3 retries

\vspace{1em}

$3 \times 3 \times 3 = 27$ attempts per failure

\pause

\vspace{2em}

100 req/s = 2,700 attempts/s

\end{frame}
```

Here's another nightmare scenario - retry storms. You have a payment processing service. Frontend talks to API Gateway talks to Payment Service. Each layer has 3 retries. That means one failed request becomes 3 times 3 times 3 - 27 attempts! [pause] At scale, if you're handling 100 requests per second, that's 2,700 attempts per second hitting your already stressed service. The service that's struggling gets MORE load, not less. Death spiral begins.

```{=latex}
\begin{frame}
\frametitle{Exponential Back-off}

The solution

\end{frame}
```

Part 2: Exponential Back-off with Jitter

```{=latex}
\begin{frame}
\frametitle{Linear vs Exponential}

Linear: 1s, 2s, 3s, 4s, 5s...

\vspace{2em}

Exponential: 1s, 2s, 4s, 8s, 16s...

\vspace{2em}

Wait time grows with severity

\end{frame}
```

Why does exponential backoff work better than linear? With linear backoff, you wait 1 second, 2 seconds, 3, 4, 5... With exponential, it's 1, 2, 4, 8, 16... See the difference? Exponential backoff reduces load quickly as failures continue. It gives the service real time to recover. It's self-regulating - worse problems automatically get longer waits. The key insight: wait time should grow with the severity of the problem.

```{=latex}
\begin{frame}[fragile]
\frametitle{Basic Exponential Back-off Implementation}

\begin{verbatim}
import time
import random

def retry_with_backoff(func, max_retries=5):
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            
            # Exponential backoff: 2^attempt seconds
            wait_time = 2 ** attempt
            time.sleep(wait_time)
    
    raise Exception("Max retries exceeded")
\end{verbatim}

Wait times: 1s, 2s, 4s, 8s, 16s

\end{frame}
```

Here's a basic implementation. We try the function, and if it fails, we wait 2 to the power of attempt seconds. So wait times are 1 second, 2 seconds, 4, 8, 16... But there's still a problem here. Everyone's still synchronized - they all retry at exactly the same times.

```{=latex}
\begin{frame}[fragile]
\frametitle{Jitter is Critical}

Without jitter: Everyone at T=4s

\vspace{1em}

With jitter:
\begin{itemize}
\item Client A: 3.2s
\item Client B: 1.8s  
\item Client C: 3.9s
\end{itemize}

\vspace{2em}

\begin{verbatim}
wait_time = random.uniform(0, 2 ** attempt)
\end{verbatim}

\end{frame}
```

This is where jitter comes in - it's absolutely critical. Without jitter, all clients retry at exactly 1 second, 2 seconds, 4 seconds. Everyone synchronized. With jitter, we spread them out across time windows. This full jitter algorithm picks a random time between 0 and our calculated backoff. So instead of everyone hitting at exactly 4 seconds, Client A might retry at 3.2 seconds, Client B at 1.8, Client C at 3.9. The load spreads out naturally. This simple randomization prevents the thundering herd.

```{=latex}
\begin{frame}[fragile]
\frametitle{Production-Ready with Tenacity}

\begin{verbatim}
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type
)

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(
        multiplier=1,
        min=1,
        max=300  # Max 5 minutes!
    ),
    retry=retry_if_exception_type(requests.RequestException)
)
def fetch_data(url):
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    return response.json()
\end{verbatim}

\end{frame}
```

For production, use a library like tenacity. Look at this configuration carefully. We stop after 5 attempts. We use exponential backoff with a minimum of 1 second. But here's the critical part - max equals 300 seconds. That's 5 minutes! This is the parameter that saved World 2 in our story. Remember this number.

```{=latex}
\begin{frame}
\frametitle{The Maximum Matters}

\textbf{World 1:} 1 second max = 3 hour outage

\vspace{2em}

\textbf{World 2:} 10 minute max = nobody noticed

\vspace{2em}

Sweet spot: \textbf{1-10 minutes}

\vspace{2em}

"Human timescales"

\end{frame}
```

Remember our recipe site story? 1 second maximum backoff led to a 3-hour complete outage that made the news. 10 minute maximum led to minor degradation that nobody even noticed. The key principle: set your maximum to 'human time frames'. Less than a minute is too short - services can't recover from real problems. More than 30 minutes is too long - users give up anyway. The sweet spot is 1 to 10 minutes. Even the most entitled customer can be mollified by support for 5 minutes while the system recovers. This one parameter can be the difference between a minor blip and a major incident.

```{=latex}
\begin{frame}
\frametitle{Strategic Giving Up}

The counterintuitive part

\end{frame}
```

Part 3: Strategic Giving Up - this is the counterintuitive part.

```{=latex}
\begin{frame}
\frametitle{Why Give Up?}

Persistent retrying:
\begin{itemize}
\item Service can't recover
\item Resources tied up
\item Cascading failures
\end{itemize}

\vspace{2em}

Strategic abandonment:
\begin{itemize}
\item Quick failure = quick recovery
\item Partial service > no service
\end{itemize}

\end{frame}
```

Why is giving up sometimes good? When you persistently retry, the dying service gets no chance to recover. Resources are tied up in doomed requests. Failures cascade to healthy services. User experience degrades for everyone. But with strategic abandonment? Quick failure equals quick recovery. Resources stay available for healthy operations. Partial service is better than no service. You get clear signals about system health. It's counterintuitive but true - sometimes the best thing you can do for a struggling system is to stop trying to use it.

```{=latex}
\begin{frame}
\frametitle{Circuit Breaker}

Three states:

\vspace{1em}

\textbf{Closed:} Normal operation

\vspace{1em}

\textbf{Open:} Reject immediately 

\vspace{1em}

\textbf{Half-Open:} Test recovery

\vspace{2em}

Like an electrical circuit breaker

\end{frame}
```

The circuit breaker pattern works like an electrical circuit breaker - it stops damage before it spreads. Three states: Closed means normal operation, requests flow through. Open means too many failures detected, we reject requests immediately without even trying. Half-open means we're testing if the service recovered with limited traffic. The benefits? Fail fast when service is down. Automatic recovery detection. And it prevents cascade failures.

```{=latex}
\begin{frame}[fragile]
\frametitle{Circuit Breaker with PyBreaker}

\begin{verbatim}
from pybreaker import CircuitBreaker

# Configure the circuit breaker
db_breaker = CircuitBreaker(
    fail_max=5,                # Open after 5 failures
    reset_timeout=60,           # Try half-open after 60s
    exclude=[KeyError]          # Don't count app errors
)

@db_breaker
def get_user_data(user_id):
    # This might fail and trip the breaker
    return database.query(f"SELECT * FROM users 
                          WHERE id={user_id}")

# Usage
try:
    user = get_user_data(123)
except Exception:
    # Breaker is open, use fallback
    user = {"id": 123, "name": "Guest User"}
\end{verbatim}

\end{frame}
```

Here's how to implement it with pybreaker. We open the circuit after 5 failures. We try half-open after 60 seconds. Important detail - we exclude KeyError. Don't trip the breaker on application bugs, only infrastructure failures. When the breaker is open, we use a fallback - Guest User. Degraded service is better than no service.

```{=latex}
\begin{frame}
\frametitle{Load Shedding}

Priority levels:

\vspace{1em}

1. Critical: Payments, auth
2. Important: Updates, search  
3. Nice-to-have: Recommendations
4. Background: Reports

\vspace{2em}

Drop less important work first

\end{frame}
```

Load shedding means choosing what to drop when you're overwhelmed. Not all requests are equal! Critical operations like payment processing and authentication must continue. User updates and searches are important but not critical. Nice-to-haves like analytics and ML recommendations can wait. Background jobs and reports go first. At 70% capacity, delay background jobs. At 80%, disable recommendations. At 90%, simplify search - no ML, just basic keyword matching. At 95%, go read-only. Our recipe site could have shed ML recommendations and kept basic search running. Users would still find recipes, just not personalized ones.

```{=latex}
\begin{frame}[fragile]
\frametitle{Load Shedding Decorator}

\begin{verbatim}
import functools
from datetime import datetime, timedelta

def optional_feature(fallback_value=None, 
                     timeout=1.0):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if system_load() > 0.8:
                # Shed this feature under load
                return fallback_value
            
            try:
                # Try with aggressive timeout
                with timeout_context(timeout):
                    return func(*args, **kwargs)
            except (TimeoutError, Exception):
                return fallback_value
        return wrapper
    return decorator

@optional_feature(fallback_value=[], timeout=0.5)
def get_recommendations(user_id):
    # This can be dropped when system is stressed
    return ml_model.predict(user_id)
\end{verbatim}

\end{frame}
```

Here's a practical decorator for optional features. If system load is above 80%, we immediately return the fallback value - don't even try. We use aggressive timeouts - half a second here. Fail fast under load. The get_recommendations function is marked as optional. When the system is stressed, users get an empty list instead of personalized recommendations. They can still use the site.

```{=latex}
\begin{frame}
\frametitle{Putting It Together}

Configuration time

\end{frame}
```

Part 4: Putting It All Together

```{=latex}
\begin{frame}[fragile]
\frametitle{Real Configuration}

\begin{verbatim}
retry:
  max_delay: 300  # 5 minutes!
  jitter: true
    
circuit_breaker:
  failure_threshold: 5
  recovery_timeout: 60
    
load_shedding:
  0.7: ["reports"]
  0.8: ["recommendations"]
  0.9: ["non_critical"]
\end{verbatim}

\end{frame}
```

Here's a real configuration bringing everything together. Look at that max_delay - 300 seconds, 5 minutes, human timescale! That's the lesson from our story. Jitter is enabled - always enable jitter. Circuit breaker opens after 5 failures, tries recovery after 60 seconds. Progressive load shedding based on system load - at 70% we drop reports, at 80% recommendations go, at 90% all non-critical operations. We also use different timeouts for different operations - 5 seconds default, 1 second for critical path, half a second for optional features. This configuration has saved us from countless outages.

```{=latex}
\begin{frame}
\frametitle{Monitor Everything}

Metrics:
\begin{itemize}
\item Retry success rate
\item Circuit breaker state
\item Load shed counts
\end{itemize}

\vspace{2em}

Alert when:
\begin{itemize}
\item Breaker open > 5 min
\item Success rate < 50\%
\item Load shedding active
\end{itemize}

\end{frame}
```

You need to monitor all of this. Track retry attempts and their success rates - if retries aren't succeeding, something's wrong. Monitor circuit breaker states - which services are struggling? Count load-shed requests - how much functionality are we sacrificing? Watch your P99 latencies. Alert when circuit breakers stay open too long - that service needs help. Alert when retry success rate drops below 50% - your backoff might be too aggressive or the service is really struggling. Alert when load shedding activates - you're approaching capacity limits. Your dashboards should show service health maps, retry ratios, and when load shedding kicked in. This visibility is crucial for operating resilient systems.

```{=latex}
\begin{frame}
\frametitle{Key Takeaways}

\begin{itemize}
\item Retries can make things worse

\vspace{1em}

\item Exponential backoff with jitter

\vspace{1em}

\item Max backoff: 1-10 minutes

\vspace{1em}

\item Give up strategically

\vspace{1em}

\item Partial service > No service
\end{itemize}

\vspace{2em}

\centering
\large{\textbf{Fail fast, recover faster}}

\end{frame}
```

Key takeaways: Retries can make things worse through thundering herds and retry storms. Exponential backoff with jitter is essential - spread load over time and set max to human timescales, 1 to 10 minutes! Strategic giving up enables recovery - circuit breakers prevent cascades, load shedding preserves critical functions. Partial service is better than no service. Remember: Failing fast often means recovering faster.

```{=latex}
\begin{frame}
\frametitle{Resources}

Libraries:
\begin{itemize}
\item \texttt{tenacity}
\item \texttt{pybreaker}
\item \texttt{backoff}
\end{itemize}

\vspace{2em}

Reading:
\begin{itemize}
\item ``Release It!'' - Nygard
\item Google SRE Book
\item \url{https://cobordism.com}
\end{itemize}

\end{frame}
```

Here are the Python libraries you should use. Tenacity for comprehensive retries - it handles everything we talked about. PyBreaker for circuit breakers - simple and effective. Backoff for lightweight decorator-based retries. For further reading, "Release It!" by Michael Nygard is the bible of production systems - read it. The Google SRE Book has an excellent chapter on cascading failures. Check out my blog at cobordism.com for more on this topic. Remember the three principles: Back off intelligently. Give up strategically. Monitor everything.

```{=latex}
\begin{frame}
\frametitle{Questions?}

\begin{center}
\huge{Questions?}

\vspace{2em}

\Large{Thank You!}

\vspace{2em}

\normalsize{
Moshe Zadka

\url{https://cobordism.com}

@moshezadka
}
\end{center}

\end{frame}
```

Questions? Thank you!

```{=latex}
\end{document}
```